In [1]:
import json

In [3]:
with open("./data-embeddings.json", "r") as f:
    data = json.load(f)

In [13]:
my_node = None
for node in data["nodes"]:
    if "Assignment_operator_(C++)" in node["title"]:
        print(node)
        my_node = node
        break

print(my_node)

{'id': 7789356, 'title': 'Assignment_operator_(C++)', 'label': 'Programming language topics', 'outlinks': [3713, 23015, 344807, 585336, 5195468], 'tokens': ['assignment', 'operator', 'c++', 'c++', 'programming', 'language', 'assignment', 'operator', 'codice_1', 'operator', 'used', 'assignment', 'like', 'operators', 'c++', 'overloaded', 'copy', 'assignment', 'operator', 'often', 'called', 'assignment', 'operator', 'special', 'case', 'assignment', 'operator', 'source', 'right-hand', 'side', 'destination', 'left-hand', 'side', 'class', 'type', 'one', 'special', 'member', 'functions', 'means', 'default', 'version', 'generated', 'automatically', 'compiler', 'programmer', 'declare', 'one', 'default', 'version', 'performs', 'memberwise', 'copy', 'member', 'copied', 'copy', 'assignment', 'operator', 'may', 'also', 'programmer-declared', 'compiler-generated', 'copy', 'assignment', 'operator', 'differs', 'copy', 'constructor', 'must', 'clean', 'data', 'members', 'assignment', 'target', 'correctl

In [14]:
import pandas as pd

In [16]:
df = pd.read_parquet("master_embeddings.parquet")
df.head()

,id,embedding
0,4908381,"[0.88818359375, 0.69091796875, -0.173217773437..."
1,50029920,"[-0.006866455078125, -1.2763671875, 0.50976562..."
2,57795942,"[0.250732421875, -0.53759765625, -0.4826660156..."
3,58731,"[0.787109375, 0.65478515625, -0.64794921875, -..."
4,16934252,"[-0.80908203125, -0.64404296875, 0.00125694274..."


In [19]:
len(df)

11701

In [17]:
import numpy as np
import pandas as pd

def get_top_5_similar(df, query_id):
    """
    Returns the 5 most similar IDs to the given query_id
    using cosine similarity.
    """

    # Ensure embeddings are numpy arrays
    embeddings = df['embedding'].apply(np.array)

    # Get query embedding
    query_row = df[df['id'] == query_id]
    if query_row.empty:
        raise ValueError(f"id {query_id} not found in dataframe")

    query_embedding = np.array(query_row.iloc[0]['embedding'])

    # Stack all embeddings into matrix
    matrix = np.vstack(embeddings.values)

    # Normalize embeddings
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    matrix_normalized = matrix / norms

    query_norm = query_embedding / np.linalg.norm(query_embedding)

    # Compute cosine similarity
    similarities = matrix_normalized @ query_norm

    # Add similarity column
    df_temp = df.copy()
    df_temp['similarity'] = similarities

    # Remove the query itself
    df_temp = df_temp[df_temp['id'] != query_id]

    # Return top 5 most similar ids
    top5 = df_temp.sort_values('similarity', ascending=False).head(5)

    return top5[['id', 'similarity']]


In [23]:
ids = get_top_5_similar(df, "7789356")["id"].values

In [26]:
ids_set = set(ids)
for node in data["nodes"]:
    if str(node["id"]) in ids_set:
        print(node["title"])
        ids_set.remove(str(node["id"]))
    if not ids_set:
        break

Move_assignment_operator
Mass_assignment_vulnerability
Assignment_(computer_science)
Exception_safety
Augmented_assignment
